# POI-first feature-combination research

This notebook tests whether previously researched features add directional value first at True POI retests and, only after the POI branch fails to produce a complete candidate, in the general GC observation stream. The frozen contract is `project_docs/poi_feature_combination_research_contract_v1.md`.

The decision is the completed retest bar close and entry is the next one-minute bar open. Development is used for chronological OOF selection. Retrospective 2024 Validation is opened only for a Development-selected model and its required anchor. The 2025+ Final-test partition is never opened. Opportunity/range predictions are diagnostic and never provide trade direction. No result here authorizes live GC or MGC execution.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "CLAUDE.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Project 1 repository root")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.statistical_research.feature_combination_runner import run_feature_combination_research

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
print(f"Repository: {ROOT}")

Repository: C:\Users\abond\Desktop\WORK FILES\Systemic\Project 1


## 1. Governed execution

The runner records byte hashes and projected schemas, fixes chronological folds before loading any outcome, deduplicates POI candidates before the label loader, persists the POI verdict before the general branch can open, and rejects any materialized 2025+ row.

In [2]:
result = run_feature_combination_research(ROOT)
display(pd.Series(result.headline, name="value").to_frame())

,value
research_id,poi_feature_combination_v1
status,READY
contract_sha256,03dd2105063e30a71fb328f8573b2a0f165c80d4ec129a...
fold_manifest_sha256,1caebdbab37e126d94e4f3fdd6c81112de29fe0ea71cd2...
branch_states,"{'POI': 'FROZEN_NO_DIRECTIONAL_MODEL', 'GENERA..."
branch_failures,"{'POI': ['no_development_eligible_model'], 'GE..."
overall_state,FROZEN_NO_DIRECTIONAL_MODEL
poi_first_completed_before_general,True
general_branch_opened,True
final_test_opened,False


## 2. Population and chronological support

In [3]:
display(result.population_summary)
display(result.fold_summary[[
    "branch", "session", "fold_id", "train_date_count", "train_end",
    "embargo_dates", "assessment_date_count", "assessment_start", "assessment_end"
]])

,branch,partition,session,stage,rows,trading_dates,minimum_trade_date,maximum_trade_date
0,GENERAL,Development,London,outcome_free_predictor_population,114834,638,2021-05-24,2023-12-29
1,GENERAL,Development,New York,outcome_free_predictor_population,191396,638,2021-05-24,2023-12-29
2,POI,Development,London,outcome_free_deduplicated_population,1104,49,2021-06-03,2023-12-04
3,POI,Development,New York,outcome_free_deduplicated_population,11066,351,2021-05-26,2023-12-27
4,POI,Development,London,label_available_model_population,1104,49,2021-06-03,2023-12-04
5,POI,Development,New York,label_available_model_population,11054,351,2021-05-26,2023-12-27
6,GENERAL,Development,London,label_available_model_population,114515,638,2021-05-24,2023-12-29
7,GENERAL,Development,New York,label_available_model_population,191127,638,2021-05-24,2023-12-29


,branch,session,fold_id,train_date_count,train_end,embargo_dates,assessment_date_count,assessment_start,assessment_end
0,POI,New York,1,126,2022-04-13,2022-04-14,42,2022-04-18,2022-07-01
1,POI,New York,2,168,2022-06-30,2022-07-01,42,2022-07-05,2022-10-13
2,POI,New York,3,210,2022-10-12,2022-10-13,42,2022-10-14,2023-02-16
3,POI,New York,4,252,2023-02-14,2023-02-16,42,2023-02-17,2023-05-16
4,POI,New York,5,294,2023-05-11,2023-05-16,42,2023-05-17,2023-10-26
5,GENERAL,London,1,126,2021-11-19,2021-11-22,42,2021-11-23,2022-01-25
6,GENERAL,London,2,168,2022-01-24,2022-01-25,42,2022-01-26,2022-04-01
7,GENERAL,London,3,210,2022-03-28,2022-04-01,42,2022-04-04,2022-06-06
8,GENERAL,London,4,252,2022-06-03,2022-06-06,42,2022-06-07,2022-08-05
9,GENERAL,London,5,294,2022-08-04,2022-08-05,42,2022-08-08,2022-10-04


## 3. Directional combination evidence

Daily rank IC uses New York dates with at least ten finite prediction/target pairs; date-block stationary bootstrap is the inference unit. The model gate separately reports all finite OOF evaluation dates so sparse POI dates are not silently discarded. Additive models must beat their frozen anchor on common IC dates.

In [4]:
directional = result.model_metrics.loc[
    result.model_metrics["target_role"].eq("DIRECTIONAL"),
    [
        "branch", "session", "model_id", "partition", "observation_count",
        "evaluation_date_count", "ic_date_count", "daily_ic_mean",
        "daily_ic_ci_low", "daily_ic_ci_high", "paired_daily_ic_improvement",
        "paired_daily_ic_ci_low", "sign_accuracy", "balanced_accuracy",
        "best_ten_date_positive_evidence_share"
    ],
].sort_values(["branch", "session", "partition", "model_id"])
display(directional)

,branch,session,model_id,partition,observation_count,evaluation_date_count,ic_date_count,daily_ic_mean,daily_ic_ci_low,daily_ic_ci_high,paired_daily_ic_improvement,paired_daily_ic_ci_low,sign_accuracy,balanced_accuracy,best_ten_date_positive_evidence_share
20,GENERAL,London,GEN1_STAT15,Development OOF,90541,504,504,0.002529,-0.030589,0.035136,0.000000,0.000000,0.494842,0.497069,0.089695
21,GENERAL,London,GEN2_FES4,Development OOF,90541,504,504,-0.006586,-0.034246,0.019992,-0.009115,-0.050077,0.483975,0.488775,0.102703
22,GENERAL,London,GEN3_STAT15_FES4,Development OOF,90541,504,504,0.004942,-0.029161,0.038619,0.002413,-0.010225,0.494662,0.496187,0.094112
26,GENERAL,New York,GEN1_STAT15,Development OOF,150955,504,504,-0.013480,-0.033848,0.005345,0.000000,0.000000,0.492125,0.492552,0.125579
27,GENERAL,New York,GEN2_FES4,Development OOF,150955,504,504,0.021404,-0.000550,0.042953,0.034884,0.006768,0.498353,0.499287,0.100281
28,GENERAL,New York,GEN3_STAT15_FES4,Development OOF,150955,504,504,-0.006918,-0.026798,0.012255,0.006562,-0.001384,0.493765,0.494143,0.119855
0,POI,London,POI1_CONTEXT,Development OOF,0,0,0,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN
1,POI,London,POI2_CONTEXT_STAT15,Development OOF,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,POI,London,POI3_CONTEXT_FES4,Development OOF,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,POI,London,POI4_CONTEXT_INTERACTIONS,Development OOF,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Opportunity diagnostics

These rows answer whether the combinations forecast future 60-minute range. They are explicitly nonselectable: an opportunity forecast is not a long/short signal.

In [5]:
opportunity = result.model_metrics.loc[
    result.model_metrics["target_role"].eq("OPPORTUNITY_DIAGNOSTIC"),
    [
        "branch", "session", "model_id", "partition", "observation_count",
        "evaluation_date_count", "ic_date_count", "daily_ic_mean",
        "daily_ic_ci_low", "daily_ic_ci_high", "paired_daily_ic_improvement"
    ],
].sort_values(["branch", "session", "model_id"])
display(opportunity)

,branch,session,model_id,partition,observation_count,evaluation_date_count,ic_date_count,daily_ic_mean,daily_ic_ci_low,daily_ic_ci_high,paired_daily_ic_improvement
23,GENERAL,London,GEN1_STAT15,Development OOF,90541,504,504,0.423333,0.403467,0.444211,0.000000
24,GENERAL,London,GEN2_FES4,Development OOF,90541,504,504,0.093845,0.068845,0.119883,-0.329488
25,GENERAL,London,GEN3_STAT15_FES4,Development OOF,90541,504,504,0.419173,0.398959,0.440300,-0.004160
29,GENERAL,New York,GEN1_STAT15,Development OOF,150955,504,504,0.586480,0.568240,0.605243,0.000000
30,GENERAL,New York,GEN2_FES4,Development OOF,150955,504,504,0.033501,0.013611,0.052663,-0.552979
31,GENERAL,New York,GEN3_STAT15_FES4,Development OOF,150955,504,504,0.587493,0.569382,0.605854,0.001013
5,POI,London,POI1_CONTEXT,Development OOF,0,0,0,NaN,NaN,NaN,0.000000
6,POI,London,POI2_CONTEXT_STAT15,Development OOF,0,0,0,NaN,NaN,NaN,NaN
7,POI,London,POI3_CONTEXT_FES4,Development OOF,0,0,0,NaN,NaN,NaN,NaN
8,POI,London,POI4_CONTEXT_INTERACTIONS,Development OOF,0,0,0,NaN,NaN,NaN,NaN


## 5. Frozen selection and policy verdicts

In [6]:
display(result.selection_verdicts)
display(result.policy_thresholds)
display(result.policy_metrics.sort_values(
    ["branch", "model_id", "partition", "round_trip_cost_ticks"]
))

,branch,session,model_id,selectable,development_eligible,development_gate_failures,development_daily_ic,development_ic_ci_low,development_observation_count,development_evaluation_date_count,development_ic_date_count,development_selected,validation_opened,validation_confirmed,validation_gate_failures
0,POI,London,POI1_CONTEXT,False,False,diagnostic_only_london_poi|minimum_four_folds|...,NaN,NaN,0,0,0,False,False,False,not_opened
1,POI,London,POI2_CONTEXT_STAT15,False,False,diagnostic_only_london_poi|minimum_four_folds|...,NaN,NaN,0,0,0,False,False,False,not_opened
2,POI,London,POI3_CONTEXT_FES4,False,False,diagnostic_only_london_poi|minimum_four_folds|...,NaN,NaN,0,0,0,False,False,False,not_opened
3,POI,London,POI4_CONTEXT_INTERACTIONS,False,False,diagnostic_only_london_poi|minimum_four_folds|...,NaN,NaN,0,0,0,False,False,False,not_opened
4,POI,London,POI5_ALL,False,False,diagnostic_only_london_poi|minimum_four_folds|...,NaN,NaN,0,0,0,False,False,False,not_opened
5,POI,New York,POI1_CONTEXT,True,False,daily_ic|daily_ic_interval|sign_accuracy,0.015542,-0.038680,5790,210,161,False,False,False,not_opened
6,POI,New York,POI2_CONTEXT_STAT15,True,False,daily_ic|daily_ic_interval|sign_accuracy|paire...,-0.023000,-0.085437,5790,210,161,False,False,False,not_opened
7,POI,New York,POI3_CONTEXT_FES4,True,False,daily_ic|daily_ic_interval|sign_accuracy|paire...,-0.000694,-0.060897,5790,210,161,False,False,False,not_opened
8,POI,New York,POI4_CONTEXT_INTERACTIONS,True,False,daily_ic_interval|sign_accuracy|paired_improve...,0.023689,-0.038348,5790,210,161,False,False,False,not_opened
9,POI,New York,POI5_ALL,True,False,daily_ic|daily_ic_interval|sign_accuracy|paire...,-0.012481,-0.077097,5790,210,161,False,False,False,not_opened


,branch,session,model_id,development_oof_prediction_count,p10,p90,enabled,reason
0,POI,London,POI1_CONTEXT,0,NaN,NaN,False,no_finite_development_oof_predictions
1,POI,London,POI2_CONTEXT_STAT15,0,NaN,NaN,False,no_finite_development_oof_predictions
2,POI,London,POI3_CONTEXT_FES4,0,NaN,NaN,False,no_finite_development_oof_predictions
3,POI,London,POI4_CONTEXT_INTERACTIONS,0,NaN,NaN,False,no_finite_development_oof_predictions
4,POI,London,POI5_ALL,0,NaN,NaN,False,no_finite_development_oof_predictions
5,POI,New York,POI1_CONTEXT,5790,-1.168746,0.333043,True,enabled
6,POI,New York,POI2_CONTEXT_STAT15,5790,-1.534014,0.595144,True,enabled
7,POI,New York,POI3_CONTEXT_FES4,5790,-1.192633,0.317254,True,enabled
8,POI,New York,POI4_CONTEXT_INTERACTIONS,5790,-1.283936,0.514310,True,enabled
9,POI,New York,POI5_ALL,5790,-1.561819,0.729854,True,enabled


,eligible_date_count,trade_count,trading_date_count,mean_gross_ticks,median_gross_ticks,mean_net_ticks,median_net_ticks,net_ticks_ci_low,net_ticks_ci_high,win_rate,profit_factor,total_net_ticks,daily_net_ticks_sharpe,maximum_drawdown_ticks,best_ten_date_pnl_share,branch,model_id,partition,cost_scenario,round_trip_cost_ticks
27,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,GENERAL,FROZEN_SELECTION_UNION,Development OOF,frictionless,0.0
28,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,GENERAL,FROZEN_SELECTION_UNION,Development OOF,base,2.6
29,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,GENERAL,FROZEN_SELECTION_UNION,Development OOF,pessimistic,4.6
18,504,2773,504,-1.161918,-1.0,-1.161918,-1.0,-3.010728,0.765762,0.478904,0.928882,-3222.0,-0.870654,-4969.0,0.168596,GENERAL,GEN1_STAT15,Development OOF,frictionless,0.0
19,504,2773,504,-1.161918,-1.0,-3.761918,-3.6,-5.530715,-1.859192,0.453660,0.787752,-10431.8,-2.809238,-10612.0,0.192165,GENERAL,GEN1_STAT15,Development OOF,base,2.6
20,504,2773,504,-1.161918,-1.0,-5.761918,-5.6,-7.498906,-4.027996,0.427696,0.694259,-15977.8,-4.286023,-16059.0,0.213293,GENERAL,GEN1_STAT15,Development OOF,pessimistic,4.6
21,504,2548,496,-1.464286,-1.0,-1.464286,-1.0,-3.464463,0.420875,0.478807,0.913032,-3731.0,-1.098090,-5576.0,0.169593,GENERAL,GEN2_FES4,Development OOF,frictionless,0.0
22,504,2548,496,-1.464286,-1.0,-4.064286,-3.6,-5.984999,-2.109354,0.452904,0.776956,-10355.8,-3.047723,-11311.6,0.194519,GENERAL,GEN2_FES4,Development OOF,base,2.6
23,504,2548,496,-1.464286,-1.0,-6.064286,-5.6,-8.121714,-4.094442,0.427002,0.686544,-15451.8,-4.539242,-16273.6,0.217475,GENERAL,GEN2_FES4,Development OOF,pessimistic,4.6
24,504,2733,504,-1.436883,-1.0,-1.436883,-1.0,-3.380986,0.542456,0.480059,0.911709,-3927.0,-1.057644,-5560.0,0.183166,GENERAL,GEN3_STAT15_FES4,Development OOF,frictionless,0.0


## 6. Compact visual audit

In [7]:
plot_table = directional.loc[directional["partition"].eq("Development OOF")].copy()
plot_table["cell"] = (
    plot_table["branch"] + " | " + plot_table["session"] + " | " + plot_table["model_id"]
)
plot_table = plot_table.sort_values("daily_ic_mean")

figure_dir = ROOT / "reports/statistical_research/poi_feature_combination/v1/figures"
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / "development_directional_daily_ic.png"

fig, ax = plt.subplots(figsize=(11, max(5, 0.35 * len(plot_table))))
colors = np.where(plot_table["daily_ic_mean"] > 0, "#2f6b4f", "#a04444")
ax.barh(plot_table["cell"], plot_table["daily_ic_mean"], color=colors)
ax.axvline(0.0, color="#333333", linewidth=0.8)
ax.axvline(0.02, color="#c58b2b", linewidth=1.0, linestyle="--", label="Development IC gate")
ax.set_xlabel("Mean daily Spearman IC")
ax.set_title("Development OOF directional IC: POI-first and general combinations")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {figure_path.relative_to(ROOT)}")

Saved: reports\statistical_research\poi_feature_combination\v1\figures\development_directional_daily_ic.png


C:\Users\abond\AppData\Local\Temp\ipykernel_8584\3140834422.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Senior quant interpretation

The decisive distinction is between *reaction opportunity* and *direction*. POI levels can coincide with larger subsequent movement without yielding a cost-surviving continuation/reversal forecast. The same distinction applies to the frozen Branch B, FES, and MLAT evidence: strong future-range association does not create a direction or an executable policy.

MGC must not inherit a GC boundary-price fill assumption. A future passing GC specification would require synchronized MGC next-bar-open execution and fresh telemetry for spread, slippage, and fill containment before deployment review.

In [8]:
assert result.headline["status"] == "READY"
assert result.headline["final_test_opened"] is False
assert result.headline["mgc_test_run"] is False
assert result.headline["poi_first_completed_before_general"] is True
assert not result.access_audit["maximum_trade_date"].dropna().ge(pd.Timestamp("2025-01-01")).any()

print(f"STATUS: {result.headline['status']}")
print(f"OVERALL STATE: {result.headline['overall_state']}")
print("FINAL TEST: UNOPENED")
print("LIVE GC/MGC AUTHORIZATION: NONE")

STATUS: READY
OVERALL STATE: FROZEN_NO_DIRECTIONAL_MODEL
FINAL TEST: UNOPENED
LIVE GC/MGC AUTHORIZATION: NONE
